In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ardl import ardl_select_order

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

df = df[["ron97", "brent", "usdmyr"]]
df = df.asfreq("ME")

df.head()


,ron97,brent,usdmyr
date,,,
2020-01-31,2.5725,63.645455,4.080110
2020-02-29,2.3780,55.657000,4.159900
2020-03-31,1.9275,32.011364,4.298409
2020-04-30,1.5625,18.378500,4.352486
2020-05-31,1.6240,29.378947,4.340906


In [2]:
from statsmodels.tsa.stattools import adfuller

adf = adfuller(df["ron97"])
print("ADF p-value (RON97):", adf[1])


ADF p-value (RON97): 0.3036489048162383


In [3]:
sel = ardl_select_order(
    endog=df["ron97"],
    exog=df[["brent", "usdmyr"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print(sel.model)


In [4]:
ardl_model = sel.model.fit()
print(ardl_model.summary())


                              ARDL Model Results                              
Dep. Variable:                  ron97   No. Observations:                   71
Model:                     ARDL(2, 1)   Log Likelihood                  74.303
Method:               Conditional MLE   S.D. of innovations              0.082
Date:                Thu, 08 Jan 2026   AIC                           -136.605
Time:                        04:59:05   BIC                           -123.201
Sample:                    03-31-2020   HQIC                          -131.287
                         - 11-30-2025                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0638      0.050      1.272      0.208      -0.036       0.164
ron97.L1       1.1485      0.109     10.493      0.000       0.930       1.367
ron97.L2      -0.2971      0.078     -3.813      0.0

In [5]:
from statsmodels.tsa.ardl import UECM

# df already: ron97, brent, usdmyr; model dropped usdmyr, so we only use brent
y = df["ron97"]
X = df[["brent"]]

uecm = UECM(y, lags=2, exog=X, order={"brent": 1}, trend="c")
uecm_res = uecm.fit()

print(uecm_res.summary())


                              UECM Model Results                              
Dep. Variable:                D.ron97   No. Observations:                   71
Model:                     UECM(2, 1)   Log Likelihood                  74.303
Method:               Conditional MLE   S.D. of innovations              3.184
Date:                Thu, 08 Jan 2026   AIC                           -136.605
Time:                        04:59:08   BIC                           -123.201
Sample:                    03-31-2020   HQIC                          -131.287
                         - 11-30-2025                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0638      0.050      1.272      0.208      -0.036       0.164
ron97.L1      -0.1486      0.046     -3.230      0.002      -0.241      -0.057
brent.L1       0.0054      0.002      3.261      0.0

In [6]:
print(uecm_res.bounds_test(case=3))


BoundsTestResult
Stat: 5.42857
Upper P-value: 0.0261
Lower P-value: 0.00725
Null: No Cointegration
Alternative: Possible Cointegration



In [7]:
import numpy as np

params = ardl_model.params  # pandas Series

phi1 = params["ron97.L1"]
phi2 = params["ron97.L2"]
phi_sum = phi1 + phi2
den = 1 - phi_sum

b0 = params["brent.L0"]
b1 = params["brent.L1"]
b_sum = b0 + b1

theta_brent = b_sum / den
c_lr = params["const"] / den

print("phi_sum:", phi_sum)
print("den (1 - phi_sum):", den)
print("b_sum (brent.L0 + brent.L1):", b_sum)
print("Long-run Brent effect on RON97:", theta_brent)
print("Long-run intercept:", c_lr)



phi_sum: 0.8513502166255122
den (1 - phi_sum): 0.14864978337448775
b_sum (brent.L0 + brent.L1): 0.005437241617046276
Long-run Brent effect on RON97: 0.036577528023357016
Long-run intercept: 0.4289945576254017


In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat

res = uecm_res  

def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError("Too few observations for BG test")

    X0 = X0_full[-n_u:, :]

    U_lags = lagmat(u, maxlag=nlags, trim='both')
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
    }


In [16]:
from statsmodels.stats.diagnostic import het_breuschpagan

def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]
    X0 = sm.add_constant(X0, has_constant="add")

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
    }


In [17]:
bg_out = bg_test_ardl_aligned(res, nlags=2)
bp_out = bp_test_ardl(res)

bg_out, bp_out


({'LM stat': 66.92513968807647,
  'LM p-value': 2.9335302195726765e-15,
  'F stat': 459.9103146293812,
  'F p-value': 1.0579120719360274e-38,
  'nobs_aux': 67,
  'resid_len': 69},
 {'LM stat': 44.92990118109676,
  'LM p-value': 2.042155640269809e-11,
  'F stat': 125.06402245300997,
  'F p-value': 5.691049912895203e-17})

In [18]:
import joblib
joblib.dump(res, "../data/joblib/ardl_ron97_brent.joblib")

['../data/joblib/ardl_ron97_brent.joblib']